### 1. Librerías y Conexión

In [27]:
from pathlib import Path
import os
import pandas as pd

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")

assert DATABASE_URL is not None, "No se encontró DATABASE_URL en el archivo .env"

engine = create_engine(DATABASE_URL)

In [28]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database(), current_user, version();"))
    for row in result:
        print(row)

('neondb', 'neondb_owner', 'PostgreSQL 17.10 (6a49db4) on x86_64-pc-linux-gnu, compiled by gcc (Debian 12.2.0-14+deb12u1) 12.2.0, 64-bit')


### 2. Rutas del Datamart

In [21]:
PATH_DM = Path.cwd() / "data" / "datamart"

paths = {
    "dim_tiempo": PATH_DM / "dim_tiempo.csv",
    "dim_geografia": PATH_DM / "dim_geografia.csv",
    "dim_vivienda": PATH_DM / "dim_vivienda.csv",
    "dim_materiales_vivienda": PATH_DM / "dim_materiales_vivienda.csv",
    "dim_habitabilidad": PATH_DM / "dim_habitabilidad.csv",
    "dim_tenencia_propiedad": PATH_DM / "dim_tenencia_propiedad.csv",
    "dim_agua": PATH_DM / "dim_agua.csv",
    "dim_saneamiento": PATH_DM / "dim_saneamiento.csv",
    "dim_energia": PATH_DM / "dim_energia.csv",
    "dim_conectividad": PATH_DM / "dim_conectividad.csv",
    "fact_hogar_bienestar": PATH_DM / "fact_hogar_bienestar.csv"
}

for nombre, path in paths.items():
    print(nombre, "→", path.exists())

dim_tiempo → True
dim_geografia → True
dim_vivienda → True
dim_materiales_vivienda → True
dim_habitabilidad → True
dim_tenencia_propiedad → True
dim_agua → True
dim_saneamiento → True
dim_energia → True
dim_conectividad → True
fact_hogar_bienestar → True


In [22]:
# LEER LOS CSVs
tablas = {}

for nombre, path in paths.items():
    tablas[nombre] = pd.read_csv(path)
    print(nombre, tablas[nombre].shape)

dim_tiempo (51, 4)
dim_geografia (6259, 9)
dim_vivienda (8, 3)
dim_materiales_vivienda (193, 11)
dim_habitabilidad (384, 10)
dim_tenencia_propiedad (7, 3)
dim_agua (50, 10)
dim_saneamiento (10, 5)
dim_energia (15, 6)
dim_conectividad (25, 7)
fact_hogar_bienestar (27260, 61)


In [23]:
# Revisar columnas

for nombre, df in tablas.items():
    print("\n" + "=" * 70)
    print(nombre)
    print(df.columns.tolist())


dim_tiempo
['tiempo_key', 'anio', 'mes', 'periodo']

dim_geografia
['geografia_key', 'ubigeo', 'dominio_cod', 'dominio', 'estrato_cod', 'estrato', 'latitud', 'longitud', 'altitud']

dim_vivienda
['vivienda_key', 'tipo_vivienda_cod', 'tipo_vivienda']

dim_materiales_vivienda
['materiales_vivienda_key', 'material_pared_cod', 'material_pared', 'material_piso_cod', 'material_piso', 'material_techo_cod', 'material_techo', 'pared_precaria_ind', 'piso_precario_ind', 'techo_precario_ind', 'vivienda_material_precario_ind']

dim_habitabilidad
['habitabilidad_key', 'rango_habitaciones_total', 'rango_habitaciones_dormir', 'nbi_vivienda_inadecuada_ind', 'nbi_hacinamiento_ind', 'nbi_sin_servicio_higienico_ind', 'nbi_ninios_no_asisten_ind', 'nbi_alta_dependencia_ind', 'categoria_carencias', 'brecha_multidimensional_ind']

dim_tenencia_propiedad
['tenencia_propiedad_key', 'tenencia_vivienda_cod', 'tenencia_vivienda']

dim_agua
['agua_key', 'fuente_agua_cod', 'fuente_agua', 'nivel_cloro_cod', 'nivel_c

### 3. Normalizar llaves

In [24]:
key_columns = {
    "dim_tiempo": ["tiempo_key"],
    "dim_geografia": ["geografia_key"],
    "dim_vivienda": ["vivienda_key"],
    "dim_materiales_vivienda": ["materiales_vivienda_key"],
    "dim_habitabilidad": ["habitabilidad_key"],
    "dim_tenencia_propiedad": ["tenencia_propiedad_key"],
    "dim_agua": ["agua_key"],
    "dim_saneamiento": ["saneamiento_key"],
    "dim_energia": ["energia_key"],
    "dim_conectividad": ["conectividad_key"],
    "fact_hogar_bienestar": [
        "tiempo_key",
        "geografia_key",
        "vivienda_key",
        "materiales_vivienda_key",
        "habitabilidad_key",
        "tenencia_propiedad_key",
        "agua_key",
        "saneamiento_key",
        "energia_key",
        "conectividad_key"
    ]
}

for tabla, columnas in key_columns.items():
    for col in columnas:
        tablas[tabla][col] = pd.to_numeric(tablas[tabla][col], errors="raise").astype("int64")

In [25]:
fact = tablas["fact_hogar_bienestar"]

cols_numericas_fact = [
    "factor_expansion",
    "habitaciones_total",
    "habitaciones_dormir",
    "alquiler_mensual_reportado",
    "alquiler_estimado_mensual",
    "alquiler_anual_imputado",
    "alquiler_estimado_anual_imputado",
    "horas_agua_dia",
    "dias_agua_semana",
    "gasto_anual_servicios_pagado_hogar",
    "gasto_anual_servicios_donado",
    "gasto_anual_servicios_autoconsumo",
    "gasto_anual_servicios_total",
    "gasto_mensual_servicios_estimado",
    "total_carencias_bienestar_habitacional"
]

for col in cols_numericas_fact:
    if col in fact.columns:
        fact[col] = pd.to_numeric(fact[col], errors="coerce")

cols_indicadores = [
    col for col in fact.columns
    if col.endswith("_ind")
]

for col in cols_indicadores:
    fact[col] = pd.to_numeric(fact[col], errors="coerce").astype("Int64")

tablas["fact_hogar_bienestar"] = fact

### 4. Esquema de NeonDB

In [29]:
SCHEMA = "dm_enaho2025"

with engine.begin() as conn:
    conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA};"))

print(f"Schema listo: {SCHEMA}")

Schema listo: dm_enaho2025


### 4.1 Para futuros debuggeos, eliminar tablas previas si existen 

In [30]:
drop_sql = f"""
DROP TABLE IF EXISTS {SCHEMA}.fact_hogar_bienestar CASCADE;

DROP TABLE IF EXISTS {SCHEMA}.dim_tiempo CASCADE;
DROP TABLE IF EXISTS {SCHEMA}.dim_geografia CASCADE;
DROP TABLE IF EXISTS {SCHEMA}.dim_vivienda CASCADE;
DROP TABLE IF EXISTS {SCHEMA}.dim_materiales_vivienda CASCADE;
DROP TABLE IF EXISTS {SCHEMA}.dim_habitabilidad CASCADE;
DROP TABLE IF EXISTS {SCHEMA}.dim_tenencia_propiedad CASCADE;
DROP TABLE IF EXISTS {SCHEMA}.dim_agua CASCADE;
DROP TABLE IF EXISTS {SCHEMA}.dim_saneamiento CASCADE;
DROP TABLE IF EXISTS {SCHEMA}.dim_energia CASCADE;
DROP TABLE IF EXISTS {SCHEMA}.dim_conectividad CASCADE;
"""

with engine.begin() as conn:
    for statement in drop_sql.split(";"):
        statement = statement.strip()
        if statement:
            conn.execute(text(statement))

print("Tablas anteriores eliminadas.")

Tablas anteriores eliminadas.


### 5. Subir tablas a NeonDB

In [31]:
orden_carga = [
    "dim_tiempo",
    "dim_geografia",
    "dim_vivienda",
    "dim_materiales_vivienda",
    "dim_habitabilidad",
    "dim_tenencia_propiedad",
    "dim_agua",
    "dim_saneamiento",
    "dim_energia",
    "dim_conectividad",
    "fact_hogar_bienestar"
]

for nombre_tabla in orden_carga:
    df = tablas[nombre_tabla]

    print(f"Cargando {nombre_tabla} | filas: {len(df)} | columnas: {len(df.columns)}")

    df.to_sql(
        name=nombre_tabla,
        con=engine,
        schema=SCHEMA,
        if_exists="replace",
        index=False,
        chunksize=1000,
        method="multi"
    )

print("Carga completada.")

Cargando dim_tiempo | filas: 51 | columnas: 4
Cargando dim_geografia | filas: 6259 | columnas: 9
Cargando dim_vivienda | filas: 8 | columnas: 3
Cargando dim_materiales_vivienda | filas: 193 | columnas: 11
Cargando dim_habitabilidad | filas: 384 | columnas: 10
Cargando dim_tenencia_propiedad | filas: 7 | columnas: 3
Cargando dim_agua | filas: 50 | columnas: 10
Cargando dim_saneamiento | filas: 10 | columnas: 5
Cargando dim_energia | filas: 15 | columnas: 6
Cargando dim_conectividad | filas: 25 | columnas: 7
Cargando fact_hogar_bienestar | filas: 27260 | columnas: 61
Carga completada.


### 6. PRIMARY KEYS y FOREIGN KEYS

In [32]:
constraints_sql = f"""
ALTER TABLE {SCHEMA}.dim_tiempo
ADD CONSTRAINT pk_dim_tiempo PRIMARY KEY (tiempo_key);

ALTER TABLE {SCHEMA}.dim_geografia
ADD CONSTRAINT pk_dim_geografia PRIMARY KEY (geografia_key);

ALTER TABLE {SCHEMA}.dim_vivienda
ADD CONSTRAINT pk_dim_vivienda PRIMARY KEY (vivienda_key);

ALTER TABLE {SCHEMA}.dim_materiales_vivienda
ADD CONSTRAINT pk_dim_materiales_vivienda PRIMARY KEY (materiales_vivienda_key);

ALTER TABLE {SCHEMA}.dim_habitabilidad
ADD CONSTRAINT pk_dim_habitabilidad PRIMARY KEY (habitabilidad_key);

ALTER TABLE {SCHEMA}.dim_tenencia_propiedad
ADD CONSTRAINT pk_dim_tenencia_propiedad PRIMARY KEY (tenencia_propiedad_key);

ALTER TABLE {SCHEMA}.dim_agua
ADD CONSTRAINT pk_dim_agua PRIMARY KEY (agua_key);

ALTER TABLE {SCHEMA}.dim_saneamiento
ADD CONSTRAINT pk_dim_saneamiento PRIMARY KEY (saneamiento_key);

ALTER TABLE {SCHEMA}.dim_energia
ADD CONSTRAINT pk_dim_energia PRIMARY KEY (energia_key);

ALTER TABLE {SCHEMA}.dim_conectividad
ADD CONSTRAINT pk_dim_conectividad PRIMARY KEY (conectividad_key);

ALTER TABLE {SCHEMA}.fact_hogar_bienestar
ADD CONSTRAINT pk_fact_hogar_bienestar PRIMARY KEY (hogar_id);

ALTER TABLE {SCHEMA}.fact_hogar_bienestar
ADD CONSTRAINT fk_fact_tiempo
FOREIGN KEY (tiempo_key)
REFERENCES {SCHEMA}.dim_tiempo(tiempo_key);

ALTER TABLE {SCHEMA}.fact_hogar_bienestar
ADD CONSTRAINT fk_fact_geografia
FOREIGN KEY (geografia_key)
REFERENCES {SCHEMA}.dim_geografia(geografia_key);

ALTER TABLE {SCHEMA}.fact_hogar_bienestar
ADD CONSTRAINT fk_fact_vivienda
FOREIGN KEY (vivienda_key)
REFERENCES {SCHEMA}.dim_vivienda(vivienda_key);

ALTER TABLE {SCHEMA}.fact_hogar_bienestar
ADD CONSTRAINT fk_fact_materiales_vivienda
FOREIGN KEY (materiales_vivienda_key)
REFERENCES {SCHEMA}.dim_materiales_vivienda(materiales_vivienda_key);

ALTER TABLE {SCHEMA}.fact_hogar_bienestar
ADD CONSTRAINT fk_fact_habitabilidad
FOREIGN KEY (habitabilidad_key)
REFERENCES {SCHEMA}.dim_habitabilidad(habitabilidad_key);

ALTER TABLE {SCHEMA}.fact_hogar_bienestar
ADD CONSTRAINT fk_fact_tenencia_propiedad
FOREIGN KEY (tenencia_propiedad_key)
REFERENCES {SCHEMA}.dim_tenencia_propiedad(tenencia_propiedad_key);

ALTER TABLE {SCHEMA}.fact_hogar_bienestar
ADD CONSTRAINT fk_fact_agua
FOREIGN KEY (agua_key)
REFERENCES {SCHEMA}.dim_agua(agua_key);

ALTER TABLE {SCHEMA}.fact_hogar_bienestar
ADD CONSTRAINT fk_fact_saneamiento
FOREIGN KEY (saneamiento_key)
REFERENCES {SCHEMA}.dim_saneamiento(saneamiento_key);

ALTER TABLE {SCHEMA}.fact_hogar_bienestar
ADD CONSTRAINT fk_fact_energia
FOREIGN KEY (energia_key)
REFERENCES {SCHEMA}.dim_energia(energia_key);

ALTER TABLE {SCHEMA}.fact_hogar_bienestar
ADD CONSTRAINT fk_fact_conectividad
FOREIGN KEY (conectividad_key)
REFERENCES {SCHEMA}.dim_conectividad(conectividad_key);
"""

with engine.begin() as conn:
    for statement in constraints_sql.split(";"):
        statement = statement.strip()
        if statement:
            conn.execute(text(statement))

print("Primary keys y foreign keys creadas correctamente.")

Primary keys y foreign keys creadas correctamente.


### 7. ÍNDICES

In [33]:
indexes_sql = f"""
CREATE INDEX IF NOT EXISTS idx_fact_tiempo_key
ON {SCHEMA}.fact_hogar_bienestar(tiempo_key);

CREATE INDEX IF NOT EXISTS idx_fact_geografia_key
ON {SCHEMA}.fact_hogar_bienestar(geografia_key);

CREATE INDEX IF NOT EXISTS idx_fact_vivienda_key
ON {SCHEMA}.fact_hogar_bienestar(vivienda_key);

CREATE INDEX IF NOT EXISTS idx_fact_materiales_vivienda_key
ON {SCHEMA}.fact_hogar_bienestar(materiales_vivienda_key);

CREATE INDEX IF NOT EXISTS idx_fact_habitabilidad_key
ON {SCHEMA}.fact_hogar_bienestar(habitabilidad_key);

CREATE INDEX IF NOT EXISTS idx_fact_tenencia_propiedad_key
ON {SCHEMA}.fact_hogar_bienestar(tenencia_propiedad_key);

CREATE INDEX IF NOT EXISTS idx_fact_agua_key
ON {SCHEMA}.fact_hogar_bienestar(agua_key);

CREATE INDEX IF NOT EXISTS idx_fact_saneamiento_key
ON {SCHEMA}.fact_hogar_bienestar(saneamiento_key);

CREATE INDEX IF NOT EXISTS idx_fact_energia_key
ON {SCHEMA}.fact_hogar_bienestar(energia_key);

CREATE INDEX IF NOT EXISTS idx_fact_conectividad_key
ON {SCHEMA}.fact_hogar_bienestar(conectividad_key);

CREATE INDEX IF NOT EXISTS idx_fact_brecha_multidimensional
ON {SCHEMA}.fact_hogar_bienestar(brecha_multidimensional_ind);
"""

with engine.begin() as conn:
    for statement in indexes_sql.split(";"):
        statement = statement.strip()
        if statement:
            conn.execute(text(statement))

print("Índices creados correctamente.")

Índices creados correctamente.


### 8. Validaciones

In [ ]:
# FILAS CARGADAS

validacion_sql = f"""
SELECT 'dim_tiempo' AS tabla, COUNT(*) AS filas FROM {SCHEMA}.dim_tiempo
UNION ALL
SELECT 'dim_geografia', COUNT(*) FROM {SCHEMA}.dim_geografia
UNION ALL
SELECT 'dim_vivienda', COUNT(*) FROM {SCHEMA}.dim_vivienda
UNION ALL
SELECT 'dim_materiales_vivienda', COUNT(*) FROM {SCHEMA}.dim_materiales_vivienda
UNION ALL
SELECT 'dim_habitabilidad', COUNT(*) FROM {SCHEMA}.dim_habitabilidad
UNION ALL
SELECT 'dim_tenencia_propiedad', COUNT(*) FROM {SCHEMA}.dim_tenencia_propiedad
UNION ALL
SELECT 'dim_agua', COUNT(*) FROM {SCHEMA}.dim_agua
UNION ALL
SELECT 'dim_saneamiento', COUNT(*) FROM {SCHEMA}.dim_saneamiento
UNION ALL
SELECT 'dim_energia', COUNT(*) FROM {SCHEMA}.dim_energia
UNION ALL
SELECT 'dim_conectividad', COUNT(*) FROM {SCHEMA}.dim_conectividad
UNION ALL
SELECT 'fact_hogar_bienestar', COUNT(*) FROM {SCHEMA}.fact_hogar_bienestar;
"""

validacion_filas = pd.read_sql(validacion_sql, engine)
validacion_filas

,tabla,filas
0,dim_tiempo,51
1,dim_geografia,6259
2,dim_vivienda,8
3,dim_materiales_vivienda,193
4,dim_habitabilidad,384
5,dim_tenencia_propiedad,7
6,dim_agua,50
7,dim_saneamiento,10
8,dim_energia,15
9,dim_conectividad,25


In [35]:
# RELACIONES

validacion_relaciones_sql = f"""
SELECT
    COUNT(*) AS filas_fact,
    COUNT(dt.tiempo_key) AS match_tiempo,
    COUNT(dg.geografia_key) AS match_geografia,
    COUNT(dv.vivienda_key) AS match_vivienda,
    COUNT(dm.materiales_vivienda_key) AS match_materiales,
    COUNT(dh.habitabilidad_key) AS match_habitabilidad,
    COUNT(dp.tenencia_propiedad_key) AS match_tenencia,
    COUNT(da.agua_key) AS match_agua,
    COUNT(ds.saneamiento_key) AS match_saneamiento,
    COUNT(de.energia_key) AS match_energia,
    COUNT(dc.conectividad_key) AS match_conectividad
FROM {SCHEMA}.fact_hogar_bienestar f
LEFT JOIN {SCHEMA}.dim_tiempo dt
    ON f.tiempo_key = dt.tiempo_key
LEFT JOIN {SCHEMA}.dim_geografia dg
    ON f.geografia_key = dg.geografia_key
LEFT JOIN {SCHEMA}.dim_vivienda dv
    ON f.vivienda_key = dv.vivienda_key
LEFT JOIN {SCHEMA}.dim_materiales_vivienda dm
    ON f.materiales_vivienda_key = dm.materiales_vivienda_key
LEFT JOIN {SCHEMA}.dim_habitabilidad dh
    ON f.habitabilidad_key = dh.habitabilidad_key
LEFT JOIN {SCHEMA}.dim_tenencia_propiedad dp
    ON f.tenencia_propiedad_key = dp.tenencia_propiedad_key
LEFT JOIN {SCHEMA}.dim_agua da
    ON f.agua_key = da.agua_key
LEFT JOIN {SCHEMA}.dim_saneamiento ds
    ON f.saneamiento_key = ds.saneamiento_key
LEFT JOIN {SCHEMA}.dim_energia de
    ON f.energia_key = de.energia_key
LEFT JOIN {SCHEMA}.dim_conectividad dc
    ON f.conectividad_key = dc.conectividad_key;
"""

pd.read_sql(validacion_relaciones_sql, engine)

,filas_fact,match_tiempo,match_geografia,match_vivienda,match_materiales,match_habitabilidad,match_tenencia,match_agua,match_saneamiento,match_energia,match_conectividad
0,27260,27260,27260,27260,27260,27260,27260,27260,27260,27260,27260


### EJEMPLO

In [37]:
query_prueba = f"""
SELECT
    g.dominio,
    COUNT(*) AS hogares_muestra,
    ROUND(
        (
            SUM(
                f.factor_expansion::numeric 
                * COALESCE(f.brecha_multidimensional_ind, 0)::numeric
            )
            / NULLIF(SUM(f.factor_expansion::numeric), 0)
            * 100
        )::numeric,
        2
    ) AS pct_brecha_multidimensional
FROM {SCHEMA}.fact_hogar_bienestar f
JOIN {SCHEMA}.dim_geografia g
    ON f.geografia_key = g.geografia_key
GROUP BY g.dominio
ORDER BY pct_brecha_multidimensional DESC;
"""

pd.read_sql(query_prueba, engine)

,dominio,hogares_muestra,pct_brecha_multidimensional
0,Sierra Norte,1902,74.84
1,Selva,6770,68.05
2,Sierra Centro,5212,65.98
3,Sierra Sur,3569,55.34
4,Costa Norte,4234,43.44
5,Costa Centro,2523,32.11
6,Costa Sur,1560,27.22
7,Lima Metropolitana,1490,17.57
